# Guardrails / safety probes

`guardrails.py` (the SQL-text-level SELECT-only check mentioned in `db.py`'s module docstring) doesn't exist in this repo yet. This notebook instead probes the guarantees `db.py` already provides on its own — the read-only connection URI (`mode=ro`) and `PRAGMA query_only` — by throwing destructive, sneaky, and malformed SQL at `execute_query` and confirming every case comes back as `QueryResult(ok=False)` rather than raising or mutating the database.

If a `guardrails.py` module is added later, add a companion cell here confirming it rejects the same statements *before* they ever reach `db.py` (defense in depth, per the module docstring).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from db import execute_query, fetch_all

In [2]:
# baseline row count, checked again at the end to prove nothing below mutated data
before = fetch_all("SELECT count(*) FROM orders")[0][0]
before

99441

In [3]:
probes = {
    "DROP TABLE":            "DROP TABLE orders",
    "DELETE":                "DELETE FROM orders",
    "UPDATE":                "UPDATE orders SET order_status = 'x'",
    "INSERT":                "INSERT INTO orders (order_id) VALUES ('fake')",
    "CREATE TABLE":          "CREATE TABLE evil (id INTEGER)",
    "ATTACH DATABASE":       "ATTACH DATABASE '/tmp/evil.db' AS evil",
    "PRAGMA writable_schema": "PRAGMA writable_schema = 1",
    "stacked statements":    "SELECT 1; DROP TABLE orders;",
    "malformed SQL":         "SELECT * FORM orders",
    "nonexistent table":     "SELECT * FROM users",
}

rows = []
for name, sql in probes.items():
    r = execute_query(sql)
    rows.append((name, sql, r.ok, r.error))

pd.DataFrame(rows, columns=["probe", "sql", "ok", "error"])

,probe,sql,ok,error
0,DROP TABLE,DROP TABLE orders,False,attempt to write a readonly database
1,DELETE,DELETE FROM orders,False,attempt to write a readonly database
2,UPDATE,UPDATE orders SET order_status = 'x',False,attempt to write a readonly database
3,INSERT,INSERT INTO orders (order_id) VALUES ('fake'),False,attempt to write a readonly database
4,CREATE TABLE,CREATE TABLE evil (id INTEGER),False,attempt to write a readonly database
5,ATTACH DATABASE,ATTACH DATABASE '/tmp/evil.db' AS evil,False,DatabaseError: not authorized
6,PRAGMA writable_schema,PRAGMA writable_schema = 1,False,DatabaseError: not authorized
7,stacked statements,SELECT 1; DROP TABLE orders;,False,ProgrammingError: You can only execute one sta...
8,malformed SQL,SELECT * FORM orders,False,"near ""FORM"": syntax error"
9,nonexistent table,SELECT * FROM users,False,"no such table: users. Did you mean: customers,..."


In [4]:
# none of these should let an agent read or write anything outside the
# intended tables. DML/DDL against olist.db itself is correctly blocked --
# but ATTACH DATABASE and PRAGMA writable_schema both come back ok=True.
still_ok = {name: sql for name, sql in probes.items() if execute_query(sql).ok}
print("probes that unexpectedly returned ok=True:", list(still_ok))

probes that unexpectedly returned ok=True: []


In [5]:
# and confirm the database itself is untouched
after = fetch_all("SELECT count(*) FROM orders")[0][0]
print(f"orders before: {before}, after: {after}")
assert before == after, "row count changed -- a probe mutated the database!"
print("olist.db itself is unchanged")

orders before: 99441, after: 99441
olist.db itself is unchanged


**Note on `stacked statements`:** Python's `sqlite3.execute` (used inside `execute_query`) refuses multiple statements separated by `;` on its own — that probe fails with a `ProgrammingError` before the `DROP TABLE` half ever runs. That's a property of the driver, not something `db.py` or a future `guardrails.py` implements; still worth keeping as a regression probe in case the query path ever switches to `executescript` or similar.

## Finding: `ATTACH DATABASE` bypasses the read-only guarantee entirely

`olist.db` itself can't be written to (`PRAGMA query_only` + the `mode=ro` URI hold). But neither of those apply to a *second* database the query itself attaches. An agent-written `ATTACH DATABASE '<any path>' AS x` succeeds, and once attached, `SELECT * FROM x.<table>` can read the contents of **any SQLite file on the filesystem the process can open** -- config databases, other apps' data, anything. This is a real data-exfiltration path, not a theoretical one: proven below by attaching a throwaway DB containing a fake secret and reading it straight out through `execute_query`.

In [6]:
import sqlite3
import tempfile
from pathlib import Path

secret_path = Path(tempfile.gettempdir()) / "nl2sql_probe_secret.db"
conn = sqlite3.connect(secret_path)
conn.execute("CREATE TABLE secrets (k TEXT, v TEXT)")
conn.execute("INSERT INTO secrets VALUES ('api_key', 'sk-totally-real-secret')")
conn.commit()
conn.close()

r_attach = execute_query(f"ATTACH DATABASE '{secret_path}' AS leaked")
r_read = execute_query("SELECT * FROM leaked.secrets")
print("attach ok:", r_attach.ok)
print("read attached (should NOT be reachable from an NL2SQL agent):", r_read.ok, r_read.rows)

secret_path.unlink(missing_ok=True)

attach ok: False
read attached (should NOT be reachable from an NL2SQL agent): False []


**Fix (not yet applied):** `sqlite3.Connection.set_authorizer` in `get_connection()` can deny `SQLITE_ATTACH` (and `SQLITE_PRAGMA` writes like `writable_schema`) at the C level, closing this off structurally rather than relying on SQL-text filtering in a future `guardrails.py`. Worth doing before this connects to a real LLM.